In [1]:
from trials_pipeline import fetch_trials, extract_fields, run_pipeline, build_include_exclude_patterns, classify
import pandas as pd

In [2]:
df1 = pd.read_csv("rare_disease_2020-01-01_to_2023-12-31_trials.csv", keep_default_na=False)
df1["date_window"] = "2020-01-01 to 2023-12-31"

df2 = pd.read_csv("rare_disease_2023-07-01_to_2026-07-01_trials.csv", keep_default_na=False)
df2["date_window"] = "2023-07-01 to 2026-07-01"

df_all = pd.concat([df1, df2], ignore_index=True)
df_all = df_all[df_all["phase"] != "NA"]

In [3]:
lookup = pd.read_csv("industry_sponsor_normalization_USABLE_LOOKUP.csv")
df_all["lead_sponsor_normalized"] = df_all["lead_sponsor"].str.strip().str.lower()
lookup["raw_sponsor_name_normalized"] = lookup["raw_sponsor_name"].str.strip().str.lower()
df_all = df_all.merge(
    lookup[["raw_sponsor_name_normalized", "dashboard_sponsor_name"]],
    left_on="lead_sponsor_normalized",
    right_on="raw_sponsor_name_normalized",
    how="left",
    indicator=True
)
df_all["dashboard_sponsor_name"] = df_all["dashboard_sponsor_name"].fillna(df_all["lead_sponsor"])

In [4]:
from trials_pipeline import build_include_exclude_patterns, classify

mesh_dict = pd.read_csv("mesh_include_exclude_dictionary.csv")
print(mesh_dict["Disease Area"].unique())  # confirm 'rare_disease' shows up before proceeding

mesh_dict = mesh_dict[mesh_dict["Disease Area"] == "rare_disease"]

rare_disease_include_keywords = {
    "Generic": ["rare disease", "rare diseases"],
    "ALS": ["amyotrophic lateral sclerosis", "als"],
    "Huntington": ["huntington disease", "huntington's disease"],
    "PAH": ["pulmonary arterial hypertension", "pah"],
    "Fabry": ["fabry disease"],
    "PKU": ["phenylketonuria", "pku"],
    "Barth": ["barth syndrome"],
    "GM1": ["gm1 gangliosidosis"],
    "Pompe": ["pompe disease"],
    "FXS": ["fragile x syndrome", "fxs"],
    "HS": ["hidradenitis suppurativa"],
    "FSGS": ["focal segmental glomerulosclerosis", "fsgs"],
    "IgAN": ["immunoglobulin a nephropathy", "iga nephropathy"],
    "Bardet-Biedl": ["bardet-biedl syndrome"],
    "Genetic obesity": ["pomc deficiency", "pcsk1 deficiency", "lepr deficiency"],
    "Krabbe": ["globoid cell", "krabbe disease"],
    "Canavan": ["canavan disease"],
    "Choroideremia": ["choroideremia"],
    "LCH": ["langerhans cell histiocytosis", "lch"],
    "ECD": ["erdheim-chester disease", "ecd"],
    "IPF": ["idiopathic pulmonary fibrosis"],
    "Waldenstrom": ["waldenstrom macroglobulinemia"],
    "Richter": ["richter transformation"],
    "Burkitt": ["burkitt lymphoma"],
    "Hairy cell": ["hairy cell leukemia"],
    "Bullous pemphigoid": ["bullous pemphigoid"],
    "Mitochondrial": ["mitochondrial disease"],
    "Autoimmune hepatitis": ["autoimmune hepatitis"],
    "Hemophilia": ["hemophilia b"],
    "MPS": ["mucopolysaccharidosis"],
    "hATTR": ["transthyretin-mediated amyloidosis", "hereditary transthyretin"],
    "Tay-Sachs": ["tay-sachs"],
    "Sandhoff": ["sandhoff disease"],
    "EGFR ex20ins": ["exon 20 insertion"],
    "BRAF V600E": ["braf v600e"],
}

rare_disease_include_keywords["Additional terms"] = [
    "osteogenesis imperfecta",
    "paroxysmal nocturnal hemoglobinuria",
    "glioma", "astrocytoma", "ependymoma", "medulloblastoma", "meningioma",
    "neuroectodermal tumor", "rhabdoid tumor", "choroid plexus", "pineal tumor",
    "central nervous system neoplasm",
    "merkel cell carcinoma",
    "hypophosphatemia",
    "friedreich",
    "tyrosinemia",
    "alkaptonuria",
    "maple syrup urine disease", "msud",
    "chaple disease",
    "familial chylomicronemia", "fcs",
    "haemophilia",
    "duchenne muscular dystrophy", "duchene muscular dystrophy",
    "hyperoxaluria", "ph1",
    "leptin-melanocortin",
    "homocystin",
]

include_pattern, exclude_pattern = build_include_exclude_patterns(mesh_dict, rare_disease_include_keywords)

df_all["indication_classification"] = df_all["conditions"].apply(
    lambda c: classify(c, include_pattern, exclude_pattern, match_label="true_rare_disease")
)
df_all["is_drug_biologic"] = df_all["intervention_types"].str.contains("DRUG|BIOLOGICAL", na=False)

<StringArray>
['cardiovascular',         'kidney',       'diabetes',          'liver',
        'obesity',     'autoimmune',   'rare_disease']
Length: 7, dtype: str


In [5]:
review_list = (
    df_all[df_all["indication_classification"] == "needs_review"]["conditions"]
    .str.split("|").explode().value_counts()
)
review_list.to_csv("rare_disease_conditions_needing_review.csv")

In [6]:
# 1. Phase filter
df_phased = df_all[~df_all["phase"].isin(["BLANK", "PHASE2,PHASE3"])]

# 2. Observational split + true-rare-disease-drug filter
df_observational = df_all[
    (df_all["study_type"] == "OBSERVATIONAL")
    & (df_all["indication_classification"] == "true_rare_disease")
]
df_true_rare_disease_drug = df_phased[
    (df_phased["indication_classification"] == "true_rare_disease")
    & (df_phased["is_drug_biologic"])
]

# 3. Company and indication views
company_early_phases = ["EARLY_PHASE1", "PHASE1", "PHASE1,PHASE2"]
company_late_phases = ["PHASE3", "PHASE4"]
indication_early_phases = ["EARLY_PHASE1", "PHASE1", "PHASE1,PHASE2", "PHASE2"]
indication_late_phases = ["PHASE3", "PHASE4"]

df_company_early = df_true_rare_disease_drug[df_true_rare_disease_drug["phase"].isin(company_early_phases)]
df_company_late = df_true_rare_disease_drug[df_true_rare_disease_drug["phase"].isin(company_late_phases)]

top20_companies_all_phases = (
    df_true_rare_disease_drug.groupby("dashboard_sponsor_name")["nct_id"]
    .nunique().sort_values(ascending=False).head(20)
)
top20_companies_early_phase = (
    df_company_early.groupby("dashboard_sponsor_name")["nct_id"]
    .nunique().sort_values(ascending=False).head(20)
)
top20_by_window = (
    df_true_rare_disease_drug.groupby(["date_window", "dashboard_sponsor_name"])["nct_id"]
    .nunique().reset_index()
    .sort_values(["date_window", "nct_id"], ascending=[True, False])
    .groupby("date_window").head(20)
)

df_indications = df_true_rare_disease_drug.assign(
    conditions=df_true_rare_disease_drug["conditions"].str.split("|")
).explode("conditions")

top_indications_early = (
    df_indications[df_indications["phase"].isin(indication_early_phases)]
    .groupby("conditions")["nct_id"].nunique()
    .sort_values(ascending=False).head(20)
)
top_indications_late = (
    df_indications[df_indications["phase"].isin(indication_late_phases)]
    .groupby("conditions")["nct_id"].nunique()
    .sort_values(ascending=False).head(20)
)

# 4. Exports
top20_companies_all_phases.to_csv("rare_disease_top20_companies_all_phases.csv")
top20_companies_early_phase.to_csv("rare_disease_top20_companies_early_phase.csv")
top20_by_window.to_csv("rare_disease_top20_companies_by_date_window.csv", index=False)
top_indications_early.to_csv("rare_disease_top_indications_early_phase.csv")
top_indications_late.to_csv("rare_disease_top_indications_late_phase.csv")
df_observational.to_csv("rare_disease_observational_studies.csv", index=False)

# 5. Tableau prep
df_all.to_csv("rare_disease_full_dataset_for_tableau.csv", index=False)

# 6. Collaborators explode
df_collab = df_all.assign(
    collaborators=df_all["collaborators"].str.split("|")
).explode("collaborators")
df_collab = df_collab[df_collab["collaborators"].notna() & (df_collab["collaborators"] != "")]
df_collab.to_csv("rare_disease_collaborations.csv", index=False)